# Task 3：预训练模型比较与坐标集成

本轮比较已有的 ImageNet ResNet18、ImageNet ResNet34、ImageNet ConvNeXt-Tiny，以及眼底图像预训练的 RETFound-Green；另外测试作者已训练的中央凹/视盘 EfficientNet-B3 直接迁移。采用相同的完全重复图分组与第 0 折验证。输出单模型误差、等权集成误差、误差相关性和 40 行提交候选。

预训练权重来源：[torchvision](https://docs.pytorch.org/vision/stable/models.html)、[RETFound-Green 作者仓库](https://github.com/justinengelmann/RETFound_Green)、[Fundus Image Toolbox 作者仓库](https://github.com/berenslab/fundus_image_toolbox)。权重通过 SHA-256 检查后用 `weights_only=True` 加载；不需要原项目自定义可执行代码。第三方包需要 `torch torchvision numpy pandas pillow timm>=0.9.16`。

**Kaggle**：挂载题目数据，启用 GPU；首次下载预训练权重需要 Internet。Notebook 内含完整实现，无需上传 Python 脚本。依赖未安装时可先在单独单元运行 `%pip install timm==0.9.16`。当前脚本会优先复用本地缓存。

**结果解释**：本轮仅运行一个 64/16 划分；不是五折 OOF。官方 Evaluation 公式仍未提供，暂用原图像素平均欧氏距离。最佳轮次和集成成员均参考了这 16 张验证图，集成选择存在偏差。这里不自动向比赛提交。


## 1. 通用数据、坐标、训练与导出实现

CSV 为坐标监督，XML 仅核对。完全相同的 `0039`、`0077` 放在一组。随机几何变换同步更新标签；恢复坐标时按每张图各自宽高计算。保留断点、优化器、调度器与随机数状态。训练目录要求有足够临时保存空间。


In [ ]:
"""Reproducible fovea-localization baseline; also embedded in task3.ipynb."""
from __future__ import annotations

import argparse
from collections import defaultdict
from contextlib import contextmanager
import hashlib
import json
import math
import os
from pathlib import Path
import random
import shutil
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms


def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


def error_metrics(predicted, truth, sizes):
    delta = (np.asarray(predicted, dtype=np.float64) - np.asarray(truth, dtype=np.float64)) * sizes
    distance = np.linalg.norm(delta, axis=1)
    return dict(n=len(distance), mean_distance_px=float(distance.mean()),
                median_distance_px=float(np.median(distance)),
                p90_distance_px=float(np.quantile(distance, .9)),
                mae_x_px=float(np.abs(delta[:, 0]).mean()),
                mae_y_px=float(np.abs(delta[:, 1]).mean()),
                rmse_coordinate_px=float(np.sqrt(np.mean(delta ** 2))))


def coordinate_baselines(train, valid):
    truth = valid[['nx', 'ny']].to_numpy()
    sizes = valid[['width', 'height']].to_numpy()
    result = []
    predictions = {}
    for name, center in [('normalized_mean', train[['nx', 'ny']].mean().to_numpy()),
                         ('normalized_median', train[['nx', 'ny']].median().to_numpy())]:
        prediction = np.tile(center, (len(valid), 1))
        result.append(dict(method=name, **error_metrics(prediction, truth, sizes)))
        predictions[name] = prediction
    # Resolution-specific priors expose how much information the camera geometry provides.
    centers = train.groupby(['width', 'height'])[['nx', 'ny']].mean()
    fallback = train[['nx', 'ny']].mean().to_numpy()
    prediction = np.array([centers.loc[(int(row.width), int(row.height))].to_numpy()
                           if (int(row.width), int(row.height)) in centers.index else fallback
                           for row in valid.itertuples()])
    result.append(dict(method='resolution_mean', **error_metrics(prediction, truth, sizes)))
    predictions['resolution_mean'] = prediction
    return result, predictions


def contact_sheet(data, records, path, predictions=None):
    rows = records.reset_index(drop=True)
    n = min(len(rows), 20)
    canvas = Image.new('RGB', (1120, math.ceil(n / 4) * 232), '#f3f5f7')
    draw = ImageDraw.Draw(canvas)
    font_path = Path('C:/Windows/Fonts/arial.ttf')
    font = ImageFont.truetype(str(font_path), 16) if font_path.exists() else ImageFont.load_default()
    for i, row in rows.iloc[:n].iterrows():
        with Image.open(data / row['subset'] / row.filename) as im:
            im = im.convert('RGB')
            im.thumbnail((276, 195))
            ox, oy = (i % 4) * 280 + (280 - im.width) // 2, (i // 4) * 232
            canvas.paste(im, (ox, oy))
            points = []
            if 'nx' in row and pd.notna(row.nx):
                points.append((row.nx, row.ny, '#00ffff'))
            if predictions is not None:
                points.append((*predictions[i], '#ffff00'))
            for x, y, color in points:
                cx, cy = ox + x * im.width, oy + y * im.height
                draw.ellipse((cx - 4, cy - 4, cx + 4, cy + 4), outline=color, width=2)
                draw.line((cx - 8, cy, cx + 8, cy), fill=color)
                draw.line((cx, cy - 8, cx, cy + 8), fill=color)
        draw.text(((i % 4) * 280 + 7, (i // 4) * 232 + 199),
                  f'{int(row.image_id):04} | {int(row.width)} x {int(row.height)}', font=font, fill='#15202b')
    canvas.save(path)


def audit(data, out, seed=42, folds=5, fold=0):
    out.mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(data)
    train = assign_folds(manifest[manifest.subset == 'train'], seed, folds)
    train.to_csv(out / 'split.csv', index=False)
    manifest.to_csv(out / 'manifest.csv', index=False)
    if not 0 <= fold < folds:
        raise ValueError('fold outside [0, folds).')
    development = train[train.fold != fold]
    valid = train[train.fold == fold]
    baselines, predictions = coordinate_baselines(development, valid)
    pd.DataFrame(baselines).to_csv(out / 'coordinate_baselines.csv', index=False)
    per_image = valid[['image_id', 'width', 'height', 'x', 'y']].copy()
    for name, pred in predictions.items():
        xy = pred * valid[['width', 'height']].to_numpy()
        per_image[f'{name}_x'] = xy[:, 0]
        per_image[f'{name}_y'] = xy[:, 1]
        per_image[f'{name}_distance_px'] = np.linalg.norm(xy - valid[['x', 'y']].to_numpy(), axis=1)
    per_image.to_csv(out / 'baseline_validation_predictions.csv', index=False)
    duplicates = [g.image_id.astype(int).tolist() for _, g in manifest.groupby('pixel_sha256') if len(g) > 1]
    summary = dict(train_count=len(train), test_count=int((manifest.subset == 'test').sum()),
                   unique_train_images=int(train.pixel_sha256.nunique()), exact_duplicate_groups=duplicates,
                   resolution_counts=[dict(width=int(w), height=int(h), count=len(g))
                                      for (w, h), g in train.groupby(['width', 'height'])],
                   fold_sizes={str(k): len(g) for k, g in train.groupby('fold')}, validation_fold=fold,
                   xml_max_abs_offset_px=train[['xml_center_dx', 'xml_center_dy']].abs().max().to_dict(),
                   label_source='fovea_localization_train_GT.csv; XML used only for consistency checks',
                   split_limit='Exact duplicates grouped; patient identities unavailable, so patient-level independence is unverified.',
                   metric_status='Official evaluation formula not provided; pixel Euclidean distance is provisional.',
                   coordinate_baselines=baselines)
    save_json(out / 'audit_summary.json', summary)
    # Select examples from across the ID range instead of displaying only the first camera type.
    picks = np.linspace(0, len(train) - 1, min(20, len(train))).astype(int)
    contact_sheet(data, train.iloc[picks], out / 'training_samples.png')
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)
    return manifest, train


class FoveaDataset(Dataset):
    def __init__(self, data, records, image_size, augment=False):
        self.records = records.reset_index(drop=True)
        self.image_size = image_size
        self.augment = augment
        self.images = []
        self.jitter = transforms.ColorJitter(brightness=.2, contrast=.2, saturation=.12, hue=.02)
        self.tensor = transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize([.485, .456, .406], [.229, .224, .225])])
        for row in self.records.itertuples():
            with Image.open(data / row.subset / row.filename) as im:
                self.images.append(im.convert('RGB').resize((image_size, image_size), Image.Resampling.BILINEAR))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        im = self.images[index].copy()
        point = (np.array([row.nx, row.ny], dtype=np.float32) if row['subset'] == 'train'
                 else np.array([.5, .5], dtype=np.float32))
        if self.augment:
            if random.random() < .5:
                im = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                point[0] = 1 - point[0]
            angle, scale = math.radians(random.uniform(-10, 10)), random.uniform(.92, 1.08)
            a, b = scale * math.cos(angle), -scale * math.sin(angle)
            size, center = self.image_size, self.image_size / 2
            tx, ty = random.uniform(-.035, .035) * size, random.uniform(-.035, .035) * size
            matrix = np.array([[a, b, center + tx - a * center - b * center],
                               [-b, a, center + ty + b * center - a * center], [0, 0, 1]])
            point = ((matrix @ np.array([point[0] * size, point[1] * size, 1]))[:2] / size).astype(np.float32)
            if not (point >= 0).all() or not (point <= 1).all():
                raise ValueError('Augmentation moved label outside image.')
            inverse = np.linalg.inv(matrix)[:2].reshape(-1)
            im = im.transform((size, size), Image.Transform.AFFINE, inverse,
                              resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))
            im = self.jitter(im)
        return self.tensor(im), torch.from_numpy(point), torch.tensor([row.width, row.height], dtype=torch.float32)


def pretrained_backbone(pretrained=True, explicit=None):
    model = models.resnet18(weights=None)
    if not pretrained:
        return model
    base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
    candidates = [Path(explicit)] if explicit else []
    candidates += [base.parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd().parent / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path.cwd() / 'task2/.torch/hub/checkpoints/resnet18-f37072fd.pth',
                   Path(torch.hub.get_dir()) / 'checkpoints/resnet18-f37072fd.pth']
    for path in candidates:
        if path.is_file():
            if not hashlib.sha256(path.read_bytes()).hexdigest().startswith('f37072fd'):
                raise ValueError(f'ImageNet checkpoint hash mismatch: {path}')
            model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
            print(f'ImageNet weights: {path}', flush=True)
            return model
    # Official torchvision ImageNet weights only. No eye-dataset or Task2 fine-tuned weights.
    state = models.ResNet18_Weights.IMAGENET1K_V1.get_state_dict(progress=True, check_hash=True)
    model.load_state_dict(state)
    return model


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


class FoveaHeatmap(nn.Module):
    def __init__(self, pretrained=True, weights=None):
        super().__init__()
        self.backbone = pretrained_backbone(pretrained, weights)
        self.backbone.fc = nn.Identity()
        self.d3 = DecoderBlock(512, 256, 128)
        self.d2 = DecoderBlock(128, 128, 64)
        self.d1 = DecoderBlock(64, 64, 32)
        self.head = nn.Conv2d(32, 1, 1)
        self.freeze_bn = pretrained

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_bn:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, image):
        b = self.backbone
        value = b.maxpool(b.relu(b.bn1(b.conv1(image))))
        c1 = b.layer1(value)
        c2 = b.layer2(c1)
        c3 = b.layer3(c2)
        c4 = b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@torch.inference_mode()
def predict(model, loader, device, flip_tta=False):
    model.eval()
    predictions = []
    for images, _, _ in loader:
        images = images.to(device)
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            points = decode_heatmap(model(images))
            if flip_tta:
                other = decode_heatmap(model(images.flip(-1)))
                other[:, 0] = 1 - other[:, 0]
                points = (points + other) / 2
        predictions.append(points.cpu().numpy())
    return np.concatenate(predictions)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


def train_model(data, training, validation, out, config, device, model_builder=None, dataset_builder=None):
    out.mkdir(parents=True, exist_ok=True)
    with training_lock(out):
        # Atomic checkpoint replacement temporarily keeps both the old and new files.
        if shutil.disk_usage(out).free < 600 * 1024 ** 2:
            raise RuntimeError(f'Need at least 600 MiB free for atomic checkpoints; choose another output directory: {out}')
        config_path = out / 'config.json'
        if config_path.exists() and json.loads(config_path.read_text(encoding='utf-8')) != config:
            raise ValueError(f'Configuration changed. Use a new output directory: {out}')
        save_json(config_path, config)
        seed_everything(config['seed'])
        dataset_builder = dataset_builder or FoveaDataset
        train_set = dataset_builder(data, training, config['image_size'], augment=True)
        loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True, num_workers=0)
        valid_loader = (DataLoader(dataset_builder(data, validation, config['image_size']),
                                  batch_size=config['batch_size'], shuffle=False, num_workers=0)
                        if validation is not None else None)
        model = (model_builder(config) if model_builder else FoveaHeatmap(config['pretrained'], config['weights'])).to(device)
        backbone_ids = {id(v) for v in model.backbone.parameters()}
        decoder_params = [v for v in model.parameters() if id(v) not in backbone_ids]
        optimizer = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': config['lr_backbone']},
                                       {'params': decoder_params, 'lr': config['lr_decoder']}],
                                      weight_decay=config['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.get('schedule_epochs', config['epochs']),
                                                               eta_min=config['lr_backbone'] * .05)
        scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
        start, best, best_epoch, history = 0, float('inf'), 0, []
        last_path, best_path = out / 'last.pt', out / 'best.pt'
        if last_path.exists():
            checkpoint = torch.load(last_path, map_location='cpu', weights_only=False)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            scheduler.load_state_dict(checkpoint['scheduler'])
            scaler.load_state_dict(checkpoint['scaler'])
            start, best, best_epoch, history = (checkpoint[k] for k in ['epoch', 'best', 'best_epoch', 'history'])
            random.setstate(checkpoint['python_rng'])
            np.random.set_state(checkpoint['numpy_rng'])
            torch.set_rng_state(checkpoint['torch_rng'])
            if device.type == 'cuda':
                torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
            print(f'Resume {out.name}: epoch {start}', flush=True)
        for epoch in range(start, config['epochs']):
            begin = time.perf_counter()
            model.train()
            total, seen = 0., 0
            for images, points, _ in loader:
                images, points = images.to(device), points.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
                    loss = heatmap_loss(model(images), points, config['sigma'])
                if not torch.isfinite(loss):
                    raise RuntimeError('Non-finite training loss.')
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.)
                scaler.step(optimizer)
                scaler.update()
                total += float(loss.detach()) * len(images)
                seen += len(images)
            row = dict(epoch=epoch + 1, train_heatmap_ce=total / seen, lr_backbone=optimizer.param_groups[0]['lr'])
            if valid_loader is not None:
                prediction = predict(model, valid_loader, device)
                row.update(error_metrics(prediction, validation[['nx', 'ny']].to_numpy(),
                                         validation[['width', 'height']].to_numpy()))
                score = row['mean_distance_px']
            else:
                score = 0.
            if score < best or valid_loader is None:
                best, best_epoch = score, epoch + 1
                torch.save(dict(model=model.state_dict(), epoch=best_epoch, config=config), out / 'best.tmp')
                (out / 'best.tmp').replace(best_path)
            scheduler.step()
            row['seconds'] = time.perf_counter() - begin
            history.append(row)
            checkpoint = dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                              scheduler=scheduler.state_dict(), scaler=scaler.state_dict(), epoch=epoch + 1,
                              best=best, best_epoch=best_epoch, history=history,
                              python_rng=random.getstate(), numpy_rng=np.random.get_state(),
                              torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all())
            torch.save(checkpoint, out / 'last.tmp')
            (out / 'last.tmp').replace(last_path)
            pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
            save_json(out / 'status.json', dict(state='running', epoch=epoch + 1, epochs=config['epochs'],
                                               best_epoch=best_epoch, best_mean_distance_px=best,
                                               pid=os.getpid(), updated_unix=time.time()))
            print(f'{out.name} {epoch+1:03}/{config["epochs"]} ce={row["train_heatmap_ce"]:.4f}'
                  + (f' val={score:.2f}px best={best:.2f}px@{best_epoch}' if valid_loader else '')
                  + f' {row["seconds"]:.1f}s', flush=True)
        checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model'])
        save_json(out / 'status.json', dict(state='completed', best_epoch=best_epoch,
                                           best_mean_distance_px=best if validation is not None else None,
                                           updated_unix=time.time()))
        return model, best_epoch


def export_submission(data, test, prediction, out):
    sample = pd.read_csv(data / 'sample_submission.csv')
    if list(sample.columns) != ['ImageID', 'value'] or sample.ImageID.duplicated().any():
        raise ValueError('Unexpected submission template.')
    sizes = test[['width', 'height']].to_numpy()
    if prediction.shape != (len(test), 2) or not np.isfinite(prediction).all():
        raise ValueError('Invalid predictions.')
    xy = np.clip(prediction, 0, 1) * sizes
    xy = np.minimum(xy, sizes - 1)
    lookup = {f'{int(image_id)}_Fovea_{axis}': float(coord)
              for image_id, coords in zip(test.image_id, xy)
              for axis, coord in zip(['X', 'Y'], coords)}
    if set(sample.ImageID) != set(lookup) or len(sample) != 2 * len(test):
        raise ValueError('Template IDs do not exactly match test images.')
    sample['value'] = sample.ImageID.map(lookup)
    if sample.value.isna().any():
        raise ValueError('Missing submission coordinates.')
    sample.to_csv(out / 'submission.csv', index=False, float_format='%.6f')
    check = pd.read_csv(out / 'submission.csv')
    if not check.ImageID.equals(sample.ImageID) or not np.isfinite(check.value).all():
        raise AssertionError('Submission round trip failed.')
    raw = test[['image_id', 'width', 'height']].copy()
    raw['Fovea_X'], raw['Fovea_Y'] = xy[:, 0], xy[:, 1]
    raw.to_csv(out / 'test_predictions.csv', index=False)
    contact_sheet(data, test, out / 'test_predictions.png', prediction)


def run(data_dir=None, output_dir='task3_baseline_seed42', phase='validate', seed=42,
        folds=5, fold=0, epochs=80, image_size=320, batch_size=8, pretrained=True,
        weights=None, allow_cpu=False, lr_backbone=1e-4, lr_decoder=1e-3,
        weight_decay=1e-4, sigma=1.5):
    data, out = find_data(data_dir), Path(output_dir).resolve()
    if phase not in {'audit', 'validate', 'full'}:
        raise ValueError(f'Unknown phase: {phase}')
    out.mkdir(parents=True, exist_ok=True)
    protocol = dict(seed=seed, folds=folds, fold=fold, epochs=epochs, image_size=image_size,
                    batch_size=batch_size, pretrained=pretrained, weights=str(weights) if weights else None,
                    lr_backbone=lr_backbone, lr_decoder=lr_decoder, weight_decay=weight_decay, sigma=sigma)
    protocol_path = out / 'protocol.json'
    if protocol_path.exists() and json.loads(protocol_path.read_text(encoding='utf-8')) != protocol:
        raise ValueError('Existing output uses a different protocol; choose a new output directory.')
    save_json(protocol_path, protocol)
    manifest, train = audit(data, out, seed, folds, fold)
    if phase == 'audit':
        return out
    if image_size < 64 or image_size % 32 or epochs < 1:
        raise ValueError('image_size must be a positive multiple of 32 >=64; epochs >=1.')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cpu' and not allow_cpu:
        raise RuntimeError('CUDA unavailable; explicitly set allow_cpu=True for a small smoke check.')
    development, valid = train[train.fold != fold], train[train.fold == fold]
    fingerprint = hashlib.sha256(train.to_csv(index=False).encode()).hexdigest()
    config = dict(seed=seed, folds=folds, fold=fold, epochs=epochs, image_size=image_size,
                  batch_size=batch_size, pretrained=pretrained, weights=str(weights) if weights else None,
                  lr_backbone=lr_backbone, lr_decoder=lr_decoder, weight_decay=weight_decay,
                  sigma=sigma, dataset_fingerprint=fingerprint, selection_metric='mean_distance_px',
                  architecture='resnet18_unet_stride4', decoder='peak_local_expectation_radius3',
                  augmentation='hflip_p0.5_affine10deg_scale0.92-1.08_translate0.035_colorjitter',
                  metric_status='provisional_until_official_evaluation_is_provided')
    save_json(out / 'environment.json', dict(torch=torch.__version__, numpy=np.__version__,
                                            device=str(device), gpu=torch.cuda.get_device_name() if device.type == 'cuda' else None))
    validation_dir = out / f'fold{fold}'
    model, best_epoch = train_model(data, development, valid, validation_dir, config, device)
    development_loader = DataLoader(FoveaDataset(data, development, image_size),
                                    batch_size=batch_size, shuffle=False)
    development_prediction = predict(model, development_loader, device)
    development_metrics = error_metrics(development_prediction, development[['nx', 'ny']].to_numpy(),
                                        development[['width', 'height']].to_numpy())
    save_json(out / 'development_train_metrics.json', dict(**development_metrics,
              evaluation='eval mode, no augmentation, no TTA, same selected checkpoint as validation'))
    loader = DataLoader(FoveaDataset(data, valid, image_size), batch_size=batch_size, shuffle=False)
    scores, predictions = coordinate_baselines(development, valid)
    for name, use_tta in [('heatmap', False), ('heatmap_flip_tta', True)]:
        prediction = predict(model, loader, device, use_tta)
        scores.append(dict(method=name, **error_metrics(prediction, valid[['nx', 'ny']].to_numpy(),
                                                        valid[['width', 'height']].to_numpy())))
        predictions[name] = prediction
    table = pd.DataFrame(scores)
    table.to_csv(out / 'validation_comparison.csv', index=False)
    chosen = min(['heatmap', 'heatmap_flip_tta'], key=lambda name: float(table.set_index('method').loc[name, 'mean_distance_px']))
    selected = predictions[chosen]
    per_image = valid[['image_id', 'width', 'height', 'x', 'y']].copy()
    for name, pred in predictions.items():
        xy = pred * valid[['width', 'height']].to_numpy()
        per_image[f'{name}_x'], per_image[f'{name}_y'] = xy[:, 0], xy[:, 1]
        per_image[f'{name}_distance_px'] = np.linalg.norm(xy - valid[['x', 'y']].to_numpy(), axis=1)
    per_image.to_csv(out / 'validation_predictions.csv', index=False)
    contact_sheet(data, valid, out / 'validation_predictions.png', selected)
    selection = dict(best_epoch=best_epoch, prediction=chosen, validation_n=len(valid),
                     selection_metric='mean_distance_px',
                     limitation='One holdout used for epoch and TTA selection; these are development scores, not independent test scores.',
                     official_metric_status='not_provided')
    save_json(out / 'selection.json', selection)
    print(table.to_string(index=False), flush=True)
    if phase == 'full':
        # Reinitialize from ImageNet and train on all 80 images for the holdout-selected epoch count.
        full_config = dict(config, epochs=best_epoch, schedule_epochs=config['epochs'],
                           fold='full', selected_from_fold=fold)
        model, _ = train_model(data, train, None, out / 'fulltrain', full_config, device)
        test = manifest[manifest.subset == 'test'].copy()
        loader = DataLoader(FoveaDataset(data, test, image_size), batch_size=batch_size, shuffle=False)
        prediction = predict(model, loader, device, chosen.endswith('tta'))
        export_submission(data, test, prediction, out)
        save_json(out / 'submission_provenance.json', dict(**selection, fulltrain_images=len(train),
                                                         fulltrain_epochs=best_epoch, test_images=len(test),
                                                         cosine_schedule_epochs=config['epochs'],
                                                         submitted_to_competition=False))
    return out


## 2. 多种预训练编码器和集成实现

ResNet34 与 ConvNeXt 使用 320×320；RETFound-Green 使用作者推荐的 392×392，并将各通道 mean/std 设为 0.5。三个模型都接热图定位头；Green 使用多个 Transformer 层的局部特征。骨干/定位头学习率：CNN 为 1e-4/1e-3，Green 为 1e-5/1e-3。

训练均为 80 轮，按原图验证距离选最佳轮次。全量阶段从预训练权重重新开始，保留 80 轮的余弦周期，只执行所选轮数。主要比较均不使用 TTA。

把各模型的最终坐标等权平均，枚举非空模型子集，并保存所有结果。没有在 16 张图上连续优化融合权重。最佳单模型与最佳等权集成均保留；若单模型更好，`submission_selected.csv` 会明确选择单模型，另行保存 `submission_ensemble.csv` 供比较。


In [ ]:
"""Compare published pretrained encoders on fixed Task3 folds, then ensemble coordinates."""
from __future__ import annotations

import argparse
import gc
import hashlib
import itertools
import json
from pathlib import Path
import shutil
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import models, transforms


TASK = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ARTIFACT_ROOT = Path(r'outputs')
if Path('/kaggle/working').exists():
    ARTIFACT_ROOT = Path('/kaggle/working/task3_artifacts')
elif not ARTIFACT_ROOT.exists():
    ARTIFACT_ROOT = TASK / 'task3_artifacts'
PRETRAINED_ROOT = ARTIFACT_ROOT / 'task3_pretrained'
DEFAULT_OUTPUT = ARTIFACT_ROOT / 'task3_runs/task3_pretrained_ensemble_seed42'
WEIGHT_URLS = {
    'resnet34': 'https://download.pytorch.org/models/resnet34-b627a593.pth',
    'convnext_tiny': 'https://download.pytorch.org/models/convnext_tiny-983f1562.pth',
    'retfound_green': 'https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth',
}

MODEL_SPECS = {
    'resnet34': dict(image_size=320, batch_size=8, lr_backbone=1e-4, lr_decoder=1e-3,
                     weights=str(TASK.parent / 'task2/.torch/hub/checkpoints/resnet34-b627a593.pth'),
                     sha256_prefix='b627a593', normalization='imagenet',
                     source='https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet34.html',
                     pretraining='ImageNet-1K, torchvision IMAGENET1K_V1'),
    'convnext_tiny': dict(image_size=320, batch_size=8, lr_backbone=1e-4, lr_decoder=1e-3,
                         weights=str(Path.home() / '.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth'),
                         sha256_prefix='983f1562', normalization='imagenet',
                         source='https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html',
                         pretraining='ImageNet-1K, torchvision IMAGENET1K_V1'),
    'retfound_green': dict(image_size=392, batch_size=4, lr_backbone=1e-5, lr_decoder=1e-3,
                          weights=str(PRETRAINED_ROOT / 'weights/retfoundgreen_statedict.pth'),
                          sha256_prefix='431de5dbc1bebbb32f60e2c0bcf8daa4f8bcbf06f7cb1e1dc97ec589713942e1',
                          normalization='half', source='https://github.com/justinengelmann/RETFound_Green',
                          pretraining='Author retinal foundation checkpoint, ViT-S/14 with registers'),
}


def require_timm():
    try:
        import timm
    except ImportError:
        local = TASK.parent / 'task2/.model_deps'
        if local.is_dir():
            sys.path.insert(0, str(local))
        import timm
    return timm


def checked_state(config):
    path = Path(config['weights'])
    if not path.is_file():
        path = PRETRAINED_ROOT / 'weights' / WEIGHT_URLS[config['model_name']].rsplit('/', 1)[-1]
        if not path.is_file():
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.hub.download_url_to_file(WEIGHT_URLS[config['model_name']], str(path),
                                           hash_prefix=config['sha256_prefix'], progress=True)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if not digest.startswith(config['sha256_prefix']):
        raise ValueError(f'Pretrained weight hash mismatch: {path}')
    return torch.load(path, map_location='cpu', weights_only=True)


class NormalizedDataset(FoveaDataset):
    def __init__(self, data, records, image_size, augment=False, normalization='imagenet'):
        super().__init__(data, records, image_size, augment)
        if normalization == 'half':
            self.tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize([.5] * 3, [.5] * 3)])


def make_dataset(config):
    def builder(data, records, image_size, augment=False):
        return NormalizedDataset(data, records, image_size, augment, config['normalization'])
    return builder


class PretrainedHeatmap(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.architecture = config['model_name']
        state = checked_state(config)
        if self.architecture == 'resnet34':
            self.backbone = models.resnet34(weights=None)
            self.backbone.load_state_dict(state, strict=True)
            self.backbone.fc = nn.Identity()
            channels = [64, 128, 256, 512]
        elif self.architecture == 'convnext_tiny':
            self.backbone = models.convnext_tiny(weights=None)
            self.backbone.load_state_dict(state, strict=True)
            self.backbone.classifier = nn.Identity()
            channels = [96, 192, 384, 768]
        elif self.architecture == 'retfound_green':
            timm = require_timm()
            self.backbone = timm.create_model('vit_small_patch14_reg4_dinov2',
                                             img_size=config['image_size'], num_classes=0, pretrained=False)
            self.backbone.load_state_dict(state, strict=True)
            self.token_fusion = nn.Sequential(nn.Conv2d(384 * 4, 128, 1, bias=False),
                                               nn.GroupNorm(8, 128), nn.ReLU(inplace=True))
            self.token_decoder = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1, bias=False),
                                                nn.GroupNorm(8, 64), nn.ReLU(inplace=True),
                                                nn.Conv2d(64, 32, 3, padding=1, bias=False),
                                                nn.GroupNorm(8, 32), nn.ReLU(inplace=True))
            self.head = nn.Conv2d(32, 1, 1)
            return
        else:
            raise ValueError(self.architecture)
        self.d3 = DecoderBlock(channels[3], channels[2], 128)
        self.d2 = DecoderBlock(128, channels[1], 64)
        self.d1 = DecoderBlock(64, channels[0], 32)
        self.head = nn.Conv2d(32, 1, 1)

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images):
        b = self.backbone
        if self.architecture == 'resnet34':
            x = b.maxpool(b.relu(b.bn1(b.conv1(images))))
            c1 = b.layer1(x)
            c2 = b.layer2(c1)
            c3 = b.layer3(c2)
            c4 = b.layer4(c3)
        elif self.architecture == 'convnext_tiny':
            features, x = [], images
            for index, block in enumerate(b.features):
                x = block(x)
                if index in [1, 3, 5, 7]:
                    features.append(x)
            c1, c2, c3, c4 = features
        else:
            maps = b.get_intermediate_layers(images, n=[2, 5, 8, 11], reshape=True, norm=True)
            x = self.token_fusion(torch.cat(maps, dim=1))
            x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
            x = self.token_decoder(x)
            x = F.interpolate(x, size=(images.shape[-2] // 4, images.shape[-1] // 4),
                              mode='bilinear', align_corners=False)
            return self.head(x)[:, 0]
        return self.head(self.d1(self.d2(self.d3(c4, c3), c2), c1))[:, 0]


def frame_predictions(records, prediction):
    frame = records[['image_id', 'width', 'height']].copy()
    if prediction.shape != (len(frame), 2) or not np.isfinite(prediction).all():
        raise ValueError('Prediction shape/values invalid.')
    frame['nx'], frame['ny'] = prediction[:, 0], prediction[:, 1]
    frame['Fovea_X'] = prediction[:, 0] * frame.width
    frame['Fovea_Y'] = prediction[:, 1] * frame.height
    return frame


def read_predictions(path, records):
    frame = pd.read_csv(path)
    if frame.image_id.duplicated().any() or set(frame.image_id) != set(records.image_id):
        raise ValueError(f'Image IDs do not match: {path}')
    frame = frame.set_index('image_id').loc[records.image_id]
    prediction = frame[['Fovea_X', 'Fovea_Y']].to_numpy() / records[['width', 'height']].to_numpy()
    if not np.isfinite(prediction).all():
        raise ValueError(f'Non-finite predictions: {path}')
    return prediction


def baseline_predictions(valid, test):
    locations = TASK / 'run_locations.json'
    baseline_root = (Path(json.loads(locations.read_text(encoding='utf-8'))['raw_run_dir'])
                     if locations.exists() else DEFAULT_OUTPUT.parent / 'task3_baseline_seed42')
    frame = pd.read_csv(baseline_root / 'validation_predictions.csv')
    if frame.image_id.duplicated().any() or set(frame.image_id) != set(valid.image_id):
        raise ValueError('Baseline validation IDs differ from ensemble validation fold.')
    frame = frame.set_index('image_id').loc[valid.image_id]
    val_pred = frame[['heatmap_x', 'heatmap_y']].to_numpy() / valid[['width', 'height']].to_numpy()
    test_pred = read_predictions(baseline_root / 'test_predictions.csv', test)
    return val_pred, test_pred


def compare_and_ensemble(out, valid, test, validation_predictions, test_predictions):
    """Predeclared equal-weight subsets; no continuous weight optimization on 16 samples."""
    truth, sizes = valid[['nx', 'ny']].to_numpy(), valid[['width', 'height']].to_numpy()
    names = sorted(validation_predictions)
    rows = []
    val_candidates, test_candidates, members = {}, {}, {}
    for name in names:
        val_candidates[name] = validation_predictions[name]
        test_candidates[name] = test_predictions[name]
        members[name] = [name]
        rows.append(dict(method=name, kind='single', members=name,
                         **error_metrics(val_candidates[name], truth, sizes)))
    for count in range(2, len(names) + 1):
        for subset in itertools.combinations(names, count):
            name = 'equal__' + '__'.join(subset)
            prediction = np.stack([validation_predictions[key] for key in subset]).mean(0)
            val_candidates[name] = prediction
            test_candidates[name] = np.stack([test_predictions[key] for key in subset]).mean(0)
            members[name] = list(subset)
            rows.append(dict(method=name, kind='equal_ensemble', members='+'.join(subset),
                             **error_metrics(prediction, truth, sizes)))
    comparison = pd.DataFrame(rows).sort_values('mean_distance_px')
    comparison.to_csv(out / 'validation_comparison.csv', index=False)
    singles = comparison[comparison.kind == 'single']
    ensembles = comparison[comparison.kind == 'equal_ensemble']
    best_single = singles.iloc[0]['method']
    best_ensemble = ensembles.iloc[0]['method']
    selected = comparison.iloc[0]['method']
    per_image = valid[['image_id', 'width', 'height', 'x', 'y']].copy()
    errors = []
    for name in names:
        delta = (validation_predictions[name] - truth) * sizes
        errors.append(delta)
        per_image[f'{name}_x'], per_image[f'{name}_y'] = (validation_predictions[name] * sizes).T
        per_image[f'{name}_distance_px'] = np.linalg.norm(delta, axis=1)
    for name, alias in [(best_ensemble, 'best_equal_ensemble'), (selected, 'selected')]:
        delta = (val_candidates[name] - truth) * sizes
        per_image[f'{alias}_x'], per_image[f'{alias}_y'] = (val_candidates[name] * sizes).T
        per_image[f'{alias}_distance_px'] = np.linalg.norm(delta, axis=1)
    per_image.to_csv(out / 'validation_predictions.csv', index=False)
    flattened = np.stack(errors).reshape(len(names), -1)
    pd.DataFrame(np.corrcoef(flattened), index=names, columns=names).to_csv(out / 'error_vector_correlation.csv')
    pair_rows = []
    for a, b in itertools.combinations(names, 2):
        da = np.linalg.norm((validation_predictions[a] - truth) * sizes, axis=1)
        db = np.linalg.norm((validation_predictions[b] - truth) * sizes, axis=1)
        pair_rows.append(dict(model_a=a, model_b=b, a_better_images=int((da < db).sum()),
                              b_better_images=int((db < da).sum()),
                              mean_prediction_disagreement_px=float(np.linalg.norm(
                                  (validation_predictions[a] - validation_predictions[b]) * sizes, axis=1).mean())))
    pd.DataFrame(pair_rows).to_csv(out / 'model_complementarity.csv', index=False)
    info = dict(selected=selected, best_single=best_single, best_equal_ensemble=best_ensemble,
                selected_members=members[selected], ensemble_members=members[best_ensemble],
                selected_weights=[1 / len(members[selected])] * len(members[selected]),
                validation_n=len(valid), candidate_count=len(comparison),
                selection_metric='mean_distance_px; provisional official metric unavailable',
                limitation='Epochs and subset selected on the same 16-image holdout; selection scores, not unbiased test estimates.',
                weighting='Equal coordinates per member, fixed candidate subsets; no continuous weight fitting.',
                submitted_to_competition=False)
    save_json(out / 'selection.json', info)
    return info, val_candidates[selected], test_candidates[selected], test_candidates[best_ensemble], comparison


def run_pretrained(model_names=None, epochs=80, output_dir=DEFAULT_OUTPUT, full=True, include_fit=True):
    out = Path(output_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is required for this experiment.')
    device = torch.device('cuda')
    data = find_data()
    manifest = load_manifest(data)
    train = assign_folds(manifest[manifest.subset == 'train'], 42, 5)
    train.to_csv(out / 'split.csv', index=False)
    development, valid = train[train.fold != 0], train[train.fold == 0]
    test = manifest[manifest.subset == 'test']
    model_names = list(model_names or MODEL_SPECS)
    protocol = dict(seed=42, folds=5, fold=0, epochs=epochs, model_names=model_names,
                    validation_metric='mean_distance_px', use_tta=False,
                    ensemble_candidates='All equal-weight subsets of available pretrained models plus baseline; no fitted weights.',
                    dataset_fingerprint=hashlib.sha256(train.to_csv(index=False).encode()).hexdigest())
    if not include_fit:
        protocol['include_fit'] = False
    protocol_path = out / 'protocol.json'
    if protocol_path.exists() and json.loads(protocol_path.read_text(encoding='utf-8')) != protocol:
        raise ValueError('Experiment protocol differs; use another output directory.')
    save_json(protocol_path, protocol)
    save_json(out / 'model_sources.json', {name: MODEL_SPECS[name] for name in model_names})
    validation_predictions, test_predictions = {}, {}
    if full:
        validation_predictions['resnet18'], test_predictions['resnet18'] = baseline_predictions(valid, test)
    for name in model_names:
        config = dict(MODEL_SPECS[name], model_name=name, seed=42, folds=5, fold=0, epochs=epochs,
                      schedule_epochs=epochs, pretrained=True, weight_decay=1e-4, sigma=1.5,
                      dataset_fingerprint=protocol['dataset_fingerprint'],
                      architecture='pretrained_encoder_heatmap_v1', selection_metric='mean_distance_px')
        config.pop('source')
        config.pop('pretraining')
        builder = make_dataset(config)
        save_json(out / 'status.json', dict(state='running', model=name, stage='validation', updated_unix=time.time()))
        model, best_epoch = train_model(data, development, valid, out / name / 'fold0', config, device,
                                        model_builder=PretrainedHeatmap, dataset_builder=builder)
        loader = DataLoader(builder(data, valid, config['image_size']), batch_size=config['batch_size'], shuffle=False)
        prediction = predict(model, loader, device)
        validation_predictions[name] = prediction
        frame_predictions(valid, prediction).to_csv(out / name / 'validation_predictions.csv', index=False)
        score = error_metrics(prediction, valid[['nx', 'ny']].to_numpy(), valid[['width', 'height']].to_numpy())
        save_json(out / name / 'validation_metrics.json', dict(best_epoch=best_epoch, **score))
        print(f'VALIDATED {name}: {score["mean_distance_px"]:.4f}px @ epoch {best_epoch}', flush=True)
        del model, loader
        gc.collect()
        torch.cuda.empty_cache()
        if full:
            save_json(out / 'status.json', dict(state='running', model=name, stage='fulltrain', updated_unix=time.time()))
            full_config = dict(config, epochs=best_epoch, fold='full', selected_from_fold=0)
            model, _ = train_model(data, train, None, out / name / 'fulltrain', full_config, device,
                                    model_builder=PretrainedHeatmap, dataset_builder=builder)
            loader = DataLoader(builder(data, test, config['image_size']), batch_size=config['batch_size'], shuffle=False)
            prediction = predict(model, loader, device)
            test_predictions[name] = prediction
            frame_predictions(test, prediction).to_csv(out / name / 'test_predictions.csv', index=False)
            del model, loader
            gc.collect()
            torch.cuda.empty_cache()
    if full:
        # Optional external zero-shot candidate is included only after its source/coordinate checks finish.
        fit_dir = out / 'fit_zeroshot'
        if include_fit and (fit_dir / 'validation_predictions.csv').is_file() and (fit_dir / 'test_predictions.csv').is_file():
            validation_predictions['fit_zeroshot'] = read_predictions(fit_dir / 'validation_predictions.csv', valid)
            test_predictions['fit_zeroshot'] = read_predictions(fit_dir / 'test_predictions.csv', test)
        info, val_pred, test_pred, ensemble_pred, comparison = compare_and_ensemble(
            out, valid, test, validation_predictions, test_predictions)
        export_submission(data, test, test_pred, out)
        shutil.copy2(out / 'submission.csv', out / 'submission_selected.csv')
        export_submission(data, test, ensemble_pred, out / 'best_ensemble' if (out / 'best_ensemble').exists()
                          else _mkdir(out / 'best_ensemble'))
        shutil.copy2(out / 'best_ensemble/submission.csv', out / 'submission_ensemble.csv')
        contact_sheet(data, valid, out / 'validation_predictions.png', val_pred)
        print(comparison.head(12).to_string(index=False), flush=True)
        print('SELECTION', json.dumps(info, ensure_ascii=False), flush=True)
    save_json(out / 'status.json', dict(state='completed', updated_unix=time.time()))
    return out


def _mkdir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path


## 3. 中央凹专用模型直接迁移

Fundus Image Toolbox 作者文档列出的训练集为 ADAM、REFUGE、IDRiD，未列 GAMMA，但未完成逐图交集核对。作为外部迁移探索记录，不宣称其绝无数据重叠。

沿用作者验证阶段的 ResizedCenterCropAndPad，根据图像边缘情况决定补边，随后裁剪并缩放到 350；输入范围 `[0,1]`。预测后逆转裁剪、补边和缩放以恢复原图坐标。该模型不使用本题标签训练；便捷推理函数裁剪流程的首次结果单独保留。


In [ ]:
"""Evaluate the author's supervised EfficientNet-B3 fovea checkpoint without Task3 training.

Reference: https://github.com/berenslab/fundus_image_toolbox (MIT).
Preprocessing follows its validation ResizedCenterCropAndPad transform. Crop/pad
offsets are inverted to return coordinates in the original nonsquare image.
"""
import hashlib
from pathlib import Path
import shutil
import tarfile

import numpy as np
from PIL import Image
import torch
from torch import nn
from torchvision import models, transforms
from torchvision.transforms import functional as TF



def crop_inverse(points, resized_width, resized_height, crop_size=350):
    top = int(round((resized_height - crop_size) / 2.))
    left = int(round((resized_width - crop_size) / 2.))
    return (np.asarray(points) * crop_size + np.array([left, top])) / np.array([resized_width, resized_height])


def validation_preprocess(image):
    """Match the author's ResizedCenterCropAndPad validation transform and record geometry."""
    width, height = image.size
    preview = TF.resize(TF.center_crop(image, height), [350, 350])
    preview = np.asarray(preview)
    black_fraction = (np.mean(preview[:10] == 0) + np.mean(preview[-10:] == 0)) / 2
    pad = black_fraction < .5
    if pad:
        resized_height, resized_width = 400, int(400 * width / height)
        image = TF.resize(image, [resized_height, resized_width])
        image = TF.pad(image, [0, 25, 0, 25], fill=0)
        side = resized_height + 50
        image = TF.resize(TF.center_crop(image, side), [350, 350])
        geometry = dict(width=resized_width, height=resized_height, side=side,
                        left=(resized_width - side) / 2, top_pad=25, padded=True)
    else:
        image = TF.resize(TF.center_crop(image, height), [350, 350])
        geometry = dict(width=width, height=height, side=height,
                        left=(width - height) / 2, top_pad=0, padded=False)
    return transforms.ToTensor()(image), geometry


def validation_inverse(points, geometry):
    return (np.asarray(points) * geometry['side'] + [geometry['left'], -geometry['top_pad']]) / [geometry['width'], geometry['height']]


def ensure_fit_weights():
    path = PRETRAINED_ROOT / 'weights/fit_efficientnet_b3.pt'
    if path.is_file():
        return path
    archive_path = path.parent / 'fit_weights.tar.gz'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not archive_path.is_file():
        torch.hub.download_url_to_file('https://zenodo.org/records/11174642/files/weights.tar.gz',
                                       str(archive_path), hash_prefix='1c6bb4e7bb82e9c7b89d573cfb3c3d86e5647aa44d547cd257a42e5aab0c27c7')
    with tarfile.open(archive_path) as archive:
        members = [member for member in archive.getmembers()
                   if member.isfile() and member.name.endswith('/multi_efficientnet-b3_best.pt')]
        if len(members) != 1:
            raise ValueError('Expected exactly one author EfficientNet-B3 checkpoint.')
        with archive.extractfile(members[0]) as source, path.open('wb') as target:
            shutil.copyfileobj(source, target)
    return path


def evaluate_fit(output_dir=DEFAULT_OUTPUT):
    torch.set_num_threads(4)
    path = ensure_fit_weights()
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == '7f2e94ac17adf9bfbfe2e994970f999ef9bcd900144bf8b6b01f98f03f4f8dc4'
    model = models.efficientnet_b3(weights=None)
    model.classifier = nn.Linear(1536, 4)
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True), strict=True)
    model.eval()
    data = find_data()
    manifest = load_manifest(data)
    train = assign_folds(manifest[manifest.subset == 'train'])
    valid, test = train[train.fold == 0], manifest[manifest.subset == 'test']
    out = Path(output_dir) / 'fit_zeroshot'
    out.mkdir(parents=True, exist_ok=True)
    # Keep the first inference-helper crop experiment; the final pipeline uses the author's validation transform.
    previous = out / 'provenance.json'
    if previous.exists() and 'ResizedCenterCropAndPad' not in previous.read_text(encoding='utf-8'):
        archive_dir = Path(output_dir) / 'fit_inference_crop_archive'
        archive_dir.mkdir(exist_ok=True)
        for file in out.glob('*.json'):
            shutil.copy2(file, archive_dir / file.name)
        for file in out.glob('*.csv'):
            shutil.copy2(file, archive_dir / file.name)
    for width, height in [(523, 350), (353, 350), (350, 350)]:
        point = np.array([.4, .55])
        left, top = int(round((width - 350) / 2)), int(round((height - 350) / 2))
        crop_point = (point * [width, height] - [left, top]) / 350
        assert np.allclose(crop_inverse(crop_point, width, height), point)
    for label, records in [('validation', valid), ('test', test)]:
        images, geometries = [], []
        for row in records.itertuples():
            with Image.open(data / row.subset / row.filename) as im:
                image, geometry = validation_preprocess(im.convert('RGB'))
            images.append(image)
            geometries.append(geometry)
        raw = []
        with torch.inference_mode():
            for start in range(0, len(images), 4):
                raw.extend(model(torch.stack(images[start:start+4]))[:, :2].numpy())
        prediction = np.stack([validation_inverse(point, geometry) for point, geometry in zip(raw, geometries)])
        frame_predictions(records, prediction).to_csv(out / f'{label}_predictions.csv', index=False)
        if label == 'validation':
            score = error_metrics(prediction, records[['nx', 'ny']].to_numpy(), records[['width', 'height']].to_numpy())
            save_json(out / 'validation_metrics.json', score)
            print('FIT zero-shot validation:', score, flush=True)
    save_json(out / 'provenance.json', dict(
        model='Author supervised EfficientNet-B3, fovea and optic disc multitask',
        source='https://github.com/berenslab/fundus_image_toolbox',
        weights_url='https://zenodo.org/records/11174642/files/weights.tar.gz',
        pretrained_datasets=['ADAM', 'REFUGE', 'IDRID'], overlap_status='No GAMMA named in author documentation; image-level overlap not verified.',
        sha256=digest, task3_training=False, normalization='ToTensor [0,1], no channel standardization',
        preprocessing='Author validation ResizedCenterCropAndPad: PIL center crop, conditional vertical pad, resize350; inverse recorded geometry',
        crop_roundtrip_checked=True, device='cpu float32'))


## 4. 配置与准备 ResNet18 对照

本地复用上一轮结果；Kaggle 若未挂载上一轮输出，会先运行完全相同的 ResNet18 基线。三种新增模型与外部迁移模型随后在同一折比较。


In [ ]:
EPOCHS = 80
MODEL_NAMES = ['resnet34', 'convnext_tiny', 'retfound_green']
USE_FIT_ZEROSHOT = True
OUTPUT_DIR = DEFAULT_OUTPUT
print('Output:', OUTPUT_DIR)

locations = TASK / 'run_locations.json'
BASELINE_DIR = (Path(json.loads(locations.read_text(encoding='utf-8'))['raw_run_dir'])
                if locations.exists() else DEFAULT_OUTPUT.parent / 'task3_baseline_seed42')
if not (BASELINE_DIR / 'test_predictions.csv').exists():
    run(output_dir=BASELINE_DIR, phase='full', epochs=80, image_size=320, batch_size=8)


In [ ]:
if USE_FIT_ZEROSHOT:
    evaluate_fit(output_dir=OUTPUT_DIR)
RESULT_DIR = run_pretrained(MODEL_NAMES, epochs=EPOCHS, output_dir=OUTPUT_DIR, full=True, include_fit=USE_FIT_ZEROSHOT)


## 5. 比较、可视化与提交

查看每张图的误差，而不仅是平均数。`model_complementarity.csv` 记录成对优劣与预测分歧，`error_vector_correlation.csv` 记录误差方向的相关性。验证可视化中青色为标签、黄色为选择的预测。

最后两份 CSV 都遵循题目 40 行模板。它们分别是验证选择的整体候选和最佳等权集成；可能相同。均未提交比赛。


In [ ]:
from IPython.display import display, Image as DisplayImage
display(pd.read_csv(RESULT_DIR / 'validation_comparison.csv'))
print((RESULT_DIR / 'selection.json').read_text(encoding='utf-8'))
display(DisplayImage(filename=str(RESULT_DIR / 'validation_predictions.png')))
display(pd.read_csv(RESULT_DIR / 'submission_selected.csv'))
print('Selected:', RESULT_DIR / 'submission_selected.csv')
print('Ensemble:', RESULT_DIR / 'submission_ensemble.csv')
